# Assignment 2 — E-Commerce Sales Analysis

Dataset: `ecommerce_orders_raw.csv`

Objective: Transform raw order data into meaningful business information using Pandas.

In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


## Part A — Data Understanding

In [ ]:
# 1. Load the dataset
df = pd.read_csv('../../data/ecommerce_orders_raw.csv')

# 2. Display its structure and dimensions
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Shape: (15100, 11)
Rows: 15100, Columns: 11


In [3]:
# 3. Identify the data types of all columns
print("Data types:")
print(df.dtypes)

Data types:
Order_ID                str
Customer_ID             str
City                    str
Product_Category        str
Product                 str
Quantity              int64
Unit_Price          float64
Discount              int64
Payment_Method          str
Order_Date              str
Rating              float64
dtype: object


In [4]:
# 4. Identify missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

Missing values per column:
Rating    401
dtype: int64


In [5]:
# 5. Identify duplicate orders
duplicates = df.duplicated().sum()
print(f"Duplicate orders: {duplicates}")

Duplicate orders: 100


In [6]:
# 6. Examine the unique values in: City, Product Category, Payment Method
for col in ['City', 'Product_Category', 'Payment_Method']:
    if col in df.columns:
        print(f"\n{col} unique values ({df[col].nunique()}):")
        print(df[col].value_counts().head(20))
    else:
        print(f"\n{col} not found in columns")


City unique values (10):
City
Chennai       1933
Pune          1880
Hyderabad     1861
Mumbai        1849
Kolkata       1846
Bangalore     1841
Coimbatore    1840
Delhi         1801
 chennai       150
CHENNAI         99
Name: count, dtype: int64

Product_Category unique values (6):
Product_Category
Clothing           2573
Books              2540
Beauty             2539
Electronics        2492
Sports             2488
Home Appliances    2468
Name: count, dtype: int64

Payment_Method unique values (5):
Payment_Method
Debit Card          3076
Cash on Delivery    3072
Credit Card         3024
UPI                 2977
Net Banking         2951
Name: count, dtype: int64


## Part B — Data Cleaning

In [7]:
# 7. Handle missing ratings
if 'Rating' in df.columns:
    missing_ratings = df['Rating'].isnull().sum()
    print(f"Missing ratings: {missing_ratings}")
    # Fill with median rating
    median_rating = df['Rating'].median()
    df['Rating'] = df['Rating'].fillna(median_rating)
    print(f"Filled missing ratings with median: {median_rating}")

Missing ratings: 401
Filled missing ratings with median: 4.0


In [8]:
# 8. Standardize city names
if 'City' in df.columns:
    print("Before standardization:")
    print(df['City'].value_counts().head(20))
    # Strip whitespace and title case
    df['City'] = df['City'].str.strip().str.title()
    print("\nAfter standardization:")
    print(df['City'].value_counts().head(20))

Before standardization:
City
Chennai       1933
Pune          1880
Hyderabad     1861
Mumbai        1849
Kolkata       1846
Bangalore     1841
Coimbatore    1840
Delhi         1801
 chennai       150
CHENNAI         99
Name: count, dtype: int64

After standardization:
City
Chennai       2182
Pune          1880
Hyderabad     1861
Mumbai        1849
Kolkata       1846
Bangalore     1841
Coimbatore    1840
Delhi         1801
Name: count, dtype: int64


In [9]:
# 9. Identify invalid quantities
if 'Quantity' in df.columns:
    invalid_qty = df[(df['Quantity'] <= 0) | (df['Quantity'].isnull())]
    print(f"Invalid quantity records: {len(invalid_qty)}")
    if len(invalid_qty) > 0:
        display(invalid_qty)
    # Handle: remove or set to 1
    df['Quantity'] = df['Quantity'].clip(lower=1)

Invalid quantity records: 20


,Order_ID,Customer_ID,City,Product_Category,Product,Quantity,Unit_Price,Discount,Payment_Method,Order_Date,Rating
101,O012623,C02458,Delhi,Home Appliances,Vacuum Cleaner,-2,17821.53,5,UPI,2026-04-30 17:12:38.690579368,5.0
476,O004601,C02136,Chennai,Clothing,Jeans,-2,96639.36,20,UPI,2025-06-26 15:38:33.514234282,4.0
1639,O006694,C01856,Pune,Sports,Football,-2,44119.66,0,Net Banking,2025-09-15 00:40:48.163210880,4.0
2420,O007460,C02632,Bangalore,Electronics,Smart Watch,-2,68963.12,0,Debit Card,2025-10-14 10:40:21.761450764,5.0
4496,O002305,C01604,Kolkata,Sports,Cricket Bat,-2,21758.55,25,Net Banking,2025-03-30 11:30:28.681912127,5.0
4861,O012185,C00664,Chennai,Books,Novel,-2,40781.24,25,UPI,2026-04-13 21:31:22.925528368,1.0
5008,O001742,C02508,Pune,Books,Self Help,-2,56782.27,20,Net Banking,2025-03-08 20:36:45.267017801,3.0
6591,O011832,C02379,Hyderabad,Electronics,Smartphone,-2,75507.40,0,Debit Card,2026-03-31 08:10:35.562370824,5.0
8972,O002382,C00456,Kolkata,Clothing,Shoes,-2,62380.34,25,Debit Card,2025-04-02 10:28:33.234215614,4.0
9230,O008580,C01669,Kolkata,Clothing,Jacket,-2,14579.25,20,Debit Card,2025-11-26 10:56:00.704046936,3.0


In [10]:
# 10. Handle duplicate orders
duplicates = df.duplicated().sum()
print(f"Duplicate orders: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicates. New shape: {df.shape}")

Duplicate orders: 100
Removed 100 duplicates. New shape: (15000, 11)


In [11]:
# 11. Convert Order_Date into an appropriate datetime format
if 'Order_Date' in df.columns:
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
    print(f"Order_Date converted to datetime")
    print(f"Date range: {df['Order_Date'].min()} to {df['Order_Date'].max()}")
    # Check for NaT
    nat_count = df['Order_Date'].isna().sum()
    if nat_count > 0:
        print(f"Warning: {nat_count} invalid dates converted to NaT")

Order_Date converted to datetime
Date range: 2025-01-01 00:00:00 to 2026-07-31 00:00:00


## Part C — Feature Engineering

In [12]:
# 12. Create: Gross_Amount = Quantity × Unit_Price
if 'Quantity' in df.columns and 'Unit_Price' in df.columns:
    df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
    print("Gross_Amount created")
    print(df['Gross_Amount'].describe())

Gross_Amount created
count     15000.000000
mean     150906.378911
std      119683.268503
min         281.300000
25%       54849.185000
50%      117142.785000
75%      226048.920000
max      499978.500000
Name: Gross_Amount, dtype: float64


In [13]:
# 13. Create: Discount_Amount = Gross_Amount × Discount / 100
if 'Gross_Amount' in df.columns and 'Discount' in df.columns:
    df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
    print("Discount_Amount created")
    print(df['Discount_Amount'].describe())

Discount_Amount created
count     15000.000000
mean      18769.355152
std       22063.266615
min           0.000000
25%        2370.869250
50%       10872.470500
75%       27094.297125
max      124845.000000
Name: Discount_Amount, dtype: float64


In [14]:
# 14. Create: Net_Amount = Gross_Amount - Discount_Amount
if 'Gross_Amount' in df.columns and 'Discount_Amount' in df.columns:
    df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']
    print("Net_Amount created")
    print(df['Net_Amount'].describe())

Net_Amount created
count     15000.000000
mean     132137.023760
std      106241.663084
min         281.300000
25%       47456.547250
50%      101197.131000
75%      195610.297500
max      498921.900000
Name: Net_Amount, dtype: float64


In [15]:
# 15. Create a Rating_Category column:
# 1–2 → Poor
# 3   → Average
# 4–5 → Good
if 'Rating' in df.columns:
    def rating_category(rating):
        if rating <= 2:
            return 'Poor'
        elif rating == 3:
            return 'Average'
        else:
            return 'Good'
    df['Rating_Category'] = df['Rating'].apply(rating_category)
    print("Rating_Category distribution:")
    print(df['Rating_Category'].value_counts())

Rating_Category distribution:
Rating_Category
Good       10583
Average     2511
Poor        1906
Name: count, dtype: int64


In [16]:
# 16. Extract the following from Order_Date:
# Year, Month, Day, Day of Week
if 'Order_Date' in df.columns:
    df['Order_Year'] = df['Order_Date'].dt.year
    df['Order_Month'] = df['Order_Date'].dt.month
    df['Order_Day'] = df['Order_Date'].dt.day
    df['Order_DayOfWeek'] = df['Order_Date'].dt.dayofweek
    print("Date components extracted: Year, Month, Day, DayOfWeek")
    print(df[['Order_Year', 'Order_Month', 'Order_Day', 'Order_DayOfWeek']].head())

Date components extracted: Year, Month, Day, DayOfWeek


   Order_Year  Order_Month  Order_Day  Order_DayOfWeek
0        2026            7         25                5
1        2026            6         27                5
2        2026            7         25                5
3        2025            2          3                0
4        2025            1         16                3


## Part D — Business Analysis

In [17]:
# 17. Calculate total revenue
if 'Net_Amount' in df.columns:
    total_revenue = df['Net_Amount'].sum()
    print(f"Total Revenue: {total_revenue:,.2f}")

Total Revenue: 1,982,055,356.40


In [18]:
# 18. Calculate total number of orders
print(f"Total number of orders: {len(df)}")

Total number of orders: 15000


In [19]:
# 19. Find the average order value
if 'Net_Amount' in df.columns:
    avg_order = df['Net_Amount'].mean()
    print(f"Average order value: {avg_order:.2f}")

Average order value: 132137.02


In [20]:
# 20. Find total revenue by product category
if 'Net_Amount' in df.columns and 'Product_Category' in df.columns:
    rev_by_cat = df.groupby('Product_Category')['Net_Amount'].sum().sort_values(ascending=False)
    print("Revenue by product category:")
    print(rev_by_cat)

Revenue by product category:


Product_Category
Clothing           3.455613e+08
Books              3.328676e+08
Beauty             3.310906e+08
Sports             3.296964e+08
Electronics        3.237187e+08
Home Appliances    3.191207e+08
Name: Net_Amount, dtype: float64


In [21]:
# 21. Find the top 10 products by revenue
if 'Net_Amount' in df.columns and 'Product' in df.columns:
    top10_products = df.groupby('Product')['Net_Amount'].sum().sort_values(ascending=False).head(10)
    print("Top 10 products by revenue:")
    print(top10_products)

Top 10 products by revenue:


Product
Jeans         9.360626e+07
Jacket        8.885343e+07
Self Help     8.838731e+07
Yoga Mat      8.832334e+07
Shampoo       8.825536e+07
Skin Serum    8.718235e+07
Laptop        8.697328e+07
Mixer         8.535969e+07
Headphones    8.360215e+07
Textbook      8.347781e+07
Name: Net_Amount, dtype: float64


In [22]:
# 22. Find the city generating the highest revenue
if 'Net_Amount' in df.columns and 'City' in df.columns:
    city_rev = df.groupby('City')['Net_Amount'].sum().sort_values(ascending=False)
    print("Revenue by city:")
    print(city_rev)
    highest_city = city_rev.index[0]
    print(f"\nCity with highest revenue: {highest_city}")

Revenue by city:
City
Chennai       2.856547e+08
Mumbai        2.478815e+08
Delhi         2.462857e+08
Coimbatore    2.462794e+08
Pune          2.439775e+08
Bangalore     2.398578e+08
Kolkata       2.393456e+08
Hyderabad     2.327731e+08
Name: Net_Amount, dtype: float64

City with highest revenue: Chennai


In [23]:
# 23. Find the most frequently used payment method
if 'Payment_Method' in df.columns:
    pm_counts = df['Payment_Method'].value_counts()
    print("Payment method distribution:")
    print(pm_counts)
    print(f"\nMost used payment method: {pm_counts.index[0]}")

Payment method distribution:
Payment_Method
Debit Card          3062
Cash on Delivery    3057
Credit Card         2998
UPI                 2952
Net Banking         2931
Name: count, dtype: int64

Most used payment method: Debit Card


In [24]:
# 24. Find the average rating for each product category
if 'Rating' in df.columns and 'Product_Category' in df.columns:
    avg_rating_cat = df.groupby('Product_Category')['Rating'].mean().sort_values(ascending=False)
    print("Average rating by product category:")
    print(avg_rating_cat)

Average rating by product category:
Product_Category
Clothing           3.909233
Beauty             3.897772
Sports             3.883279
Home Appliances    3.854523
Books              3.853803
Electronics        3.833603
Name: Rating, dtype: float64


In [25]:
# 25. Calculate monthly revenue
if 'Net_Amount' in df.columns and 'Order_Date' in df.columns:
    df['Order_YearMonth'] = df['Order_Date'].dt.to_period('M')
    monthly_rev = df.groupby('Order_YearMonth')['Net_Amount'].sum()
    print("Monthly revenue:")
    print(monthly_rev)

Monthly revenue:
Order_YearMonth
2025-01    1.080247e+08
2025-02    9.685231e+07
2025-03    1.059376e+08
2025-04    1.073704e+08
2025-05    1.063948e+08
2025-06    1.024794e+08
2025-07    1.074132e+08
2025-08    1.072570e+08
2025-09    1.083825e+08
2025-10    1.087711e+08
2025-11    1.050722e+08
2025-12    1.034943e+08
2026-01    1.108373e+08
2026-02    9.477007e+07
2026-03    1.015241e+08
2026-04    9.971387e+07
2026-05    1.064280e+08
2026-06    9.942566e+07
2026-07    1.019070e+08
Freq: M, Name: Net_Amount, dtype: float64


In [26]:
# 26. Identify the highest-revenue month
if 'Net_Amount' in df.columns and 'Order_Date' in df.columns:
    df['Order_YearMonth'] = df['Order_Date'].dt.to_period('M')
    monthly_rev = df.groupby('Order_YearMonth')['Net_Amount'].sum()
    highest_month = monthly_rev.idxmax()
    highest_rev = monthly_rev.max()
    print(f"Highest revenue month: {highest_month} ({highest_rev:,.2f})")

Highest revenue month: 2026-01 (110,837,307.04)

In [27]:
# 27. Find the top 10 customers by total spending
if 'Net_Amount' in df.columns and 'Customer_ID' in df.columns:
    top10_cust = df.groupby('Customer_ID')['Net_Amount'].sum().sort_values(ascending=False).head(10)
    print("Top 10 customers by total spending:")
    print(top10_cust)

Top 10 customers by total spending:
Customer_ID
C01527    2.398928e+06
C01856    2.308075e+06
C02170    2.179011e+06
C01626    2.177252e+06
C01882    2.104829e+06
C00990    2.057789e+06
C02988    2.009290e+06
C02588    2.002666e+06
C00139    1.988373e+06
C00944    1.971527e+06
Name: Net_Amount, dtype: float64


In [28]:
# 28. Compare revenue generated by different cities
if 'Net_Amount' in df.columns and 'City' in df.columns:
    city_rev = df.groupby('City')['Net_Amount'].sum().sort_values(ascending=False)
    print("Revenue by city:")
    print(city_rev)

Revenue by city:
City
Chennai       2.856547e+08
Mumbai        2.478815e+08
Delhi         2.462857e+08
Coimbatore    2.462794e+08
Pune          2.439775e+08
Bangalore     2.398578e+08
Kolkata       2.393456e+08
Hyderabad     2.327731e+08
Name: Net_Amount, dtype: float64


In [29]:
# 29. Find which product category has the highest average order value
if 'Net_Amount' in df.columns and 'Product_Category' in df.columns:
    avg_order_cat = df.groupby('Product_Category')['Net_Amount'].mean().sort_values(ascending=False)
    print("Average order value by product category:")
    print(avg_order_cat)
    highest_cat = avg_order_cat.index[0]
    print(f"\nCategory with highest average order value: {highest_cat}")

Average order value by product category:
Product_Category
Clothing           135196.145448
Sports             133156.859020
Books              131880.983414
Beauty             131698.727920
Electronics        130742.602972
Home Appliances    130041.049964
Name: Net_Amount, dtype: float64

Category with highest average order value: Clothing


## Summary

Assignment 2 complete. E-commerce data cleaned, features engineered, and ready for analysis.